---
title: Gemma 2 steering benchmarks
jupyter:
  jupytext:
    text_representation:
      extension: .qmd
      format_name: quarto
      format_version: '1.0'
      jupytext_version: 1.19.3
  kernelspec:
    display_name: Python 3
    name: python3
    language: python
---


This notebook clones the repo from GitHub, loads `google/gemma-2-2b`, attaches the steering hook, and runs one `lm-evaluation-harness` cell per benchmark.

Defaults use `ZeroSteerer` so you can compare against a no-op steered wrapper before swapping in a real steering method.


In [ ]:
# Clone this repo from GitHub and install it.
# Replace the placeholder URL after you create the GitHub repository.
REPO_URL = "https://github.com/JDub323/cam_enabled_steering_llm.git"
REPO_DIR = "gemma2-steering-research"

import os
if not os.path.exists(REPO_DIR):
    !git clone -q $REPO_URL
%cd $REPO_DIR
!pip install -q -e ".[eval]"

In [ ]:
# Authenticate for gated Gemma weights. Assumes you accepted the Gemma license.
from huggingface_hub import login
login()

In [ ]:
import torch
import lm_eval
from lm_eval.tasks import TaskManager

from gemma2_steering import SteeredGemma2, ZeroSteerer, ConstantSteerer, as_lm_eval_model

MODEL_NAME = "google/gemma-2-2b"
LAYER = 12
HOOK_POINT = "layer_output"  # options: layer_output, post_attention, post_mlp

model = SteeredGemma2(
    ZeroSteerer(),
    model_name=MODEL_NAME,
    layer=LAYER,
    hook_point=HOOK_POINT,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# The harness sees the underlying HF model, but the hook remains installed in-place.
lm = as_lm_eval_model(model, tokenizer=model.tokenizer, batch_size="auto")
task_manager = TaskManager(include_path="benchmarks")

In [ ]:
def run_eval(task, *, limit=20, num_fewshot=0):
    """Run one lm-eval task. Keep limit small on Colab; set limit=None for full evals."""
    return lm_eval.simple_evaluate(
        model=lm,
        tasks=[task],
        task_manager=task_manager,
        limit=limit,
        num_fewshot=num_fewshot,
        log_samples=False,
    )


def try_first_available(tasks, *, limit=20, num_fewshot=0):
    errors = {}
    for task in tasks:
        try:
            print(f"Trying {task!r}...")
            return run_eval(task, limit=limit, num_fewshot=num_fewshot)
        except Exception as exc:
            errors[task] = repr(exc)
    print("None of the candidate task names worked. Inspect installed tasks with:")
    print("!lm-eval ls tasks | grep -Ei 'hle|humanity|last_exam'")
    raise RuntimeError(errors)

In [ ]:
# Generation sanity check.
prompt = "The capital of France is"
inputs = model.tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=16)
print(model.tokenizer.decode(output[0], skip_special_tokens=True))

# Disable steering without reloading Gemma:
with model.without_steering():
    output = model.generate(**inputs, max_new_tokens=16)
print(model.tokenizer.decode(output[0], skip_special_tokens=True))

## Benchmark: MMLU

In [ ]:
# MMLU is large; use a small limit for Colab smoke tests.
mmlu_results = run_eval("mmlu", limit=20, num_fewshot=5)
mmlu_results["results"]

## Benchmark: Humanity's Last Exam

In [ ]:
# HLE availability depends on the installed lm-eval release.
# This tries common task names and prints an inspection command if unavailable.
hle_results = try_first_available(
    ["humanitys_last_exam", "humanity_last_exam", "hle", "hle_text"],
    limit=10,
    num_fewshot=0,
)
hle_results["results"]

## Benchmark: TruthfulQA

In [ ]:
truthfulqa_results = run_eval("truthfulqa_mc2", limit=50, num_fewshot=0)
truthfulqa_results["results"]

## Benchmark: Sycophancy

In [ ]:
# Local YAML in benchmarks/sycophancy_on_nlp_survey.yaml.
sycophancy_results = run_eval("sycophancy_on_nlp_survey", limit=50, num_fewshot=0)
sycophancy_results["results"]

## Benchmark: WMDP dangerous-knowledge / malicious-capability probe

In [ ]:
# WMDP covers biosecurity, cybersecurity, and chemical-security multiple-choice questions.
wmdp_results = run_eval("wmdp", limit=50, num_fewshot=0)
wmdp_results["results"]

## Swap in a constant vector

`ConstantSteerer` requires a batched vector. This is intentionally explicit; use `[1, hidden_size]` for a vector shared across the batch.

In [ ]:
hidden = model.config.hidden_size
constant = torch.zeros(1, hidden, device=model.device, dtype=torch.bfloat16)
constant[:, 0] = 1.0

model.steerer = ConstantSteerer(constant)
# Re-run any benchmark cell above to compare against ZeroSteerer.